# Bronze Replay & Backfill

Replays historical data from raw sources or API logs into Bronze tables.

**Purpose**: Backfill historical data without re-calling APIs  
**Pattern**: Replay from immutable API response logs  
**Use cases**: Data recovery, schema migration, historical analysis

In [0]:
# Backfill parameters
dbutils.widgets.dropdown("replay_mode", "api_logs", ["api_logs", "raw_files"], "Replay Mode")
dbutils.widgets.text("source_table", "", "Source Table (for api_logs mode)")
dbutils.widgets.text("source_path", "", "Source Path (for raw_files mode)")
dbutils.widgets.text("target_table", "", "Target Bronze Table")
dbutils.widgets.text("start_timestamp", "", "Start Timestamp (YYYY-MM-DD HH:MM:SS)")
dbutils.widgets.text("end_timestamp", "", "End Timestamp (YYYY-MM-DD HH:MM:SS)")
dbutils.widgets.text("batch_id", "", "New Batch ID")
dbutils.widgets.text("catalog", "workspace", "Catalog")
dbutils.widgets.text("schema", "bronze", "Schema")

# Get values
replay_mode = dbutils.widgets.get("replay_mode")
source_table = dbutils.widgets.get("source_table")
source_path = dbutils.widgets.get("source_path")
target_table = dbutils.widgets.get("target_table")
start_timestamp = dbutils.widgets.get("start_timestamp")
end_timestamp = dbutils.widgets.get("end_timestamp")
batch_id = dbutils.widgets.get("batch_id")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

if not target_table:
    raise ValueError("target_table is required")
if not batch_id:
    raise ValueError("batch_id is required for tracking")

print(f"Replay mode: {replay_mode}")
print(f"Target table: {target_table}")
print(f"Batch ID: {batch_id}")

In [0]:
from pyspark.sql import functions as F
from datetime import datetime

if replay_mode == "api_logs":
    if not source_table:
        raise ValueError("source_table is required for api_logs mode")
    
    # Read from API response logs
    print(f"Reading from {source_table}...")
    source_df = spark.table(source_table)
    
    # Apply timestamp filter if provided
    if start_timestamp:
        source_df = source_df.filter(F.col("response_timestamp") >= F.lit(start_timestamp))
    if end_timestamp:
        source_df = source_df.filter(F.col("response_timestamp") <= F.lit(end_timestamp))
    
    # Extract response body for replay
    replay_df = source_df.select(
        F.col("log_id").alias("original_log_id"),
        F.col("response_body"),
        F.col("response_timestamp"),
        F.lit(batch_id).alias("replay_batch_id"),
        F.lit(datetime.now()).alias("replay_timestamp")
    )
    
    records_to_replay = replay_df.count()
    print(f"✓ Loaded {records_to_replay} records from API logs")
    
else:
    # Mode: raw_files
    print(f"Loading from raw files at {source_path}...")
    # Placeholder for file-based replay
    # Implementation depends on file format (JSON, Parquet, etc.)
    raise NotImplementedError("raw_files mode requires custom implementation based on file format")

In [0]:
# Parse response body and prepare for Bronze
# This is a template - customize based on your API response structure

if replay_mode == "api_logs":
    # Example: Parse JSON response bodies
    from pyspark.sql.types import StructType, StructField, StringType, TimestampType
    
    # Parse JSON from response_body
    # Adjust schema based on your API structure
    parsed_df = replay_df.select(
        F.col("original_log_id"),
        F.from_json(F.col("response_body"), "string").alias("parsed_data"),  # Customize schema
        F.col("response_timestamp"),
        F.col("replay_batch_id"),
        F.col("replay_timestamp")
    )
    
    # Add Bronze metadata
    bronze_df = parsed_df.select(
        F.col("original_log_id"),
        F.col("parsed_data"),
        F.col("replay_batch_id").alias("batch_id"),
        F.col("response_timestamp").alias("source_timestamp"),
        F.col("replay_timestamp").alias("ingestion_timestamp"),
        F.lit("replay").alias("ingestion_type")
    )
    
    print(f"✓ Prepared {bronze_df.count()} records for Bronze")

In [0]:
# Append replayed data to Bronze target
if replay_mode == "api_logs":
    full_target = f"{catalog}.{schema}.{target_table}"
    
    # Append to Bronze (preserves immutability)
    bronze_df.write.mode("append").saveAsTable(full_target)
    
    print(f"✓ Replay complete: {records_to_replay} records written to {full_target}")
    print(f"  Replay batch ID: {batch_id}")
    print(f"  Time range: {start_timestamp or 'earliest'} to {end_timestamp or 'latest'}")
    
    # Summary
    print("\nReplay Summary:")
    spark.table(full_target).filter(
        F.col("batch_id") == batch_id
    ).groupBy("ingestion_type").count().show()

In [0]:
%sql
-- Verify replayed records in target table
SELECT 
    batch_id,
    ingestion_type,
    COUNT(*) as record_count,
    MIN(source_timestamp) as earliest_source,
    MAX(source_timestamp) as latest_source,
    MIN(ingestion_timestamp) as replay_start,
    MAX(ingestion_timestamp) as replay_end
FROM ${catalog}.${schema}.${target_table}
WHERE batch_id = '${batch_id}'
GROUP BY batch_id, ingestion_type;